# Introduction
This notebook is intended to visualize simulated vs. observed heads data from MODFLOW scenarios. It uses outputs generated by other notebooks, and works with structured `MODFLOW 6` grids.

This notebook requires `flopy` to read in simulated heads, and relies on `gpd`, `pandas`, and `matplotlib` for visualization and export of observed heads data already intersected with the model grid during the model build steps.

In [4]:
import os
import numpy as np, pandas as pd
from matplotlib import pyplot as plt
import geopandas as gpd
#import seaborn as sns
import flopy


# Specify Inputs and Outputs

In [ ]:
# Specify the name of the simulation and model to read
nameSim = 'Ausable'
nameModel = 'Testing_Ausable_welllogicchange'
nameModel = '%.16s'%nameModel #model name can't exceed 16 characters

# Model files directory 
dirModelFilesBase = '/Users/saminab/Documents/course/spring_2024/GLG_893/Project/GW_Modelling/Ausable/Subdirectory'
dirModelFiles= dirModelFilesBase+'/%s'%(nameSim)

# Output graphics directory -- if final folder doesn't exist it will be created
dirGraphics = '/Users/saminab/Documents/course/spring_2024/GLG_893/Project/GW_Modelling/graphic'

# Specify the inputs
nameInStructure = 'structure.yml'
nameInInputsStore = 'model_inputs.h5'
nameInInputsNumpy = 'model_inputs.npz'

## Specify the output names
nameOutHeadsPoints = 'heads_residuals_output_points.shp'

# Some variables used below
noDataVal = 1e+30

# Helper Functions

In [ ]:
def rmse(predictions, targets):
    return np.sqrt(((predictions - targets) ** 2).mean())

def mae(predictions, targets):
    return np.abs(predictions - targets).mean()

def points_to_shapefile(dfIn, outPath, outPrj=None):
    # This function takes an input points dataframe with field 'ixshapes', as generated by the grid intersection
    # and creates a point shapefile using all other attributes in that dataframe
    
    import geopandas as gpd
    
    # Rename the `ixshapes` to `geometry`
    dfOut = dfIn.copy()
    dfOut.rename(columns={'ixshapes':'geometry'}, inplace=True)

    # Drop `cellids`
    dfOut.drop(columns=['cellids'], inplace=True)
    
    # Create a geodataframe
    gdfOut = gpd.GeoDataFrame(dfOut)
    
    # Set the projection and write it out to a shapefile
    gdfOut.crs = outPrj
    gdfOut.to_file(outPath)

def load_structure(inPath):
    import yaml
    structure = yaml.load(open(inPath), Loader=yaml.Loader)
    return structure

# Read in Model Structure and Outputs
We need to read in the wells dataframe and the heads

In [ ]:
# Read in the observed heads dataframe
inInputsStore = pd.HDFStore(os.path.join(dirModelFiles,nameInInputsStore))
dfWells = inInputsStore['dfWells']


# Read in the watersheds dataframe
dfWatersheds = inInputsStore['dfWatersheds']

# Read in the drains dataframe
dfDrn = inInputsStore['dfDrn']
inInputsStore.close()

# Read in the HK and recharge rasters
loadNumpy = np.load(os.path.join(dirModelFiles,nameInInputsNumpy))
rasterHK = loadNumpy['rasterHK']
rasterRech = loadNumpy['rasterRech']

# Load the model structure
structure = load_structure(os.path.join(dirModelFiles, nameInStructure))

In [ ]:
# print dfWells['obs'] and dfWells['sim'] min and max
print('dfWells obs min: ', dfWells['obs'].min())
print('dfWells obs max: ', dfWells['obs'].max())

In [ ]:
# Get the path to the output heads file
pathOutputHeads = os.path.join(dirModelFiles, '%s.hds'%(nameModel))
pathOutputFlows = os.path.join(dirModelFiles, '%s.cbc'%(nameModel))

# Get the heads and flows object
objHeads = flopy.utils.binaryfile.HeadFile(pathOutputHeads)
objFlows = flopy.utils.binaryfile.CellBudgetFile(pathOutputFlows)

# Get data for the first timestep
headData = objHeads.get_data(totim=1.0)
flowData = objFlows.get_data(text='DRN', totim=1.0)[0] #get the first item, this is a masked array, not doing a full3d array

In [ ]:
nameInputsStore = 'model_inputs1.h5'
nameInStructure = 'structure.yml'

In [ ]:
# Load the model structure
structure = load_structure(os.path.join(dirModelFiles, nameInStructure))

In [ ]:
# Get the path to the output heads file
pathOutputHeads = os.path.join(dirModelFiles, '%s.hds'%(nameModel))
pathOutputFlows = os.path.join(dirModelFiles, '%s.cbc'%(nameModel))

# Get the heads and flows object
objHeads = flopy.utils.binaryfile.HeadFile(pathOutputHeads)
objFlows = flopy.utils.binaryfile.CellBudgetFile(pathOutputFlows)

# Get data for the first timestep
headData = objHeads.get_data(totim=1.0)
flowData = objFlows.get_data(text='DRN', totim=1.0)[0] #get the first item, this is a masked array, not doing a full3d array


In [ ]:
# print dfWell obs column min and max
print(dfWells['obs'].min())
print(dfWells['obs'].max())

In [ ]:
# Sample heads for the wells, use the second layer (to avoid dry cells--you will likely need to change this!)
extractHead = lambda x: headData[0,int(x['row']),int(x['col'])]
dfWells['sim'] = dfWells.apply(extractHead,axis=1)

# Remove erroneous values from obs and sim (below 0 and above 999)
dfWells = dfWells[(dfWells['obs'] > 0) & (dfWells['obs'] < 999)]
dfWells = dfWells[(dfWells['sim'] > 0) & (dfWells['sim'] < 999)]

# Remove wells with COUNTY == "Ottawa" from the dataframe
#dfWells = dfWells[dfWells.COUNTY != 'Ottawa']

# Compute the residuals
dfWells['resid'] = dfWells['sim'] - dfWells['obs']

# Remove wells with more than 50m of residual
#dfWells = dfWells[np.abs(dfWells['resid']) < 100]

# Add HK and recharge
extractHK = lambda x: rasterHK[0,int(x['row']),int(x['col'])]
extractRech = lambda x: rasterRech[0,int(x['row']),int(x['col'])]  
dfWells['rech'] = dfWells.apply(extractRech,axis=1)
dfWells['hk'] = dfWells.apply(extractHK,axis=1)

In [ ]:
# print min and max of dfWells['sim']
print(dfWells['sim'].min())
print(dfWells['sim'].max())
# print min and max of dfWells['obs']
print(dfWells['obs'].min())
print(dfWells['obs'].max())

In [ ]:
# mask out the values of dfWells['sim'] that are equal to noDataVal
dfWells['sim'] = np.where(dfWells['sim'] == noDataVal, np.nan, dfWells['sim'])

## Plot Sim v. Obs

In [ ]:
# Plot the simulated vs. observed heads
# Create the scatter plot
figSimbObs, axSimObs = plt.subplots()
sns.regplot(x='obs', y='sim', data=dfWells, line_kws={'color':'red'})

# Add the 1:1 line <-- change the limits here to match your data
plt.plot([170,500],[170,500], color='black', linestyle='-')

# Specify plot properties
axSimObs.set_xlabel('Observed Static Water Level (m)',fontsize=12)
axSimObs.set_ylabel('Simulated Water Level (m)',fontsize=12)
axSimObs.tick_params(axis='both', direction='in', right=True, top=True, length=6)
plt.title('Simulated vs. Observed Heads', fontsize=12)
plt.xlim([150,450]) #<-- change the limits here to match your data
plt.ylim([150,450]) #<-- change the limits here to match your data
plt.savefig(os.path.join(dirGraphics, 'sim_vs_obs_removewell.png'), dpi=300)

## Plot Residuals vs. Obs

In [ ]:
# Make the residuals plot, make the best fit line red
figResid, axResid = plt.subplots()
sns.regplot(x='obs', y='resid', data=dfWells, line_kws={'color':'red'})

# Add the 1:1 line <-- change the limits here to match your data
plt.plot([100,500],[0,0], color='black', linestyle='--')

# Specify plot properties
axResid.set_xlabel('Observed Static Water Level (m)',fontsize=12)
axResid.set_ylabel('Residual (sim-obs, m)',fontsize=12)
axResid.tick_params(axis='both', direction='in', right=True, top=True, length=6)
plt.title('Residual Plot, Sim-Obs Heads', fontsize=12)
plt.xlim([150,400]) #<-- change the limits here to match your data
plt.ylim([-150,400]) #<-- change the limits here to match your data
plt.savefig(os.path.join(dirGraphics, 'residuals.png'), dpi=300)

## Calculate some Statistics

In [ ]:
# Calculate some statistics
RMSE = rmse(dfWells['sim'],dfWells['obs'])
MAE = mae(dfWells['sim'],dfWells['obs'])

# Print these nicely
print('RMSE: %.2f m'%(RMSE))
print('MAE: %.2f m'%(MAE))

## Plot the Spatial Residuals

In [ ]:
# Get a symmetric vmax and vmin, based on min/max residuals
vmax = np.max(np.abs(dfWells['resid']))
vmin = -vmax

# Scatter plot of the residuals, spatially
fig, ax = plt.subplots()
plt.scatter(dfWells['col'],dfWells['row'],c=dfWells['resid'],cmap='viridis', vmin = -200, vmax=300)
plt.title('Residuals (sim-obs, m)/remove some wells')
plt.title('Residuals (sim-obs, m)')
# Display the colorbar, label it 
plt.colorbar(label='Residual (sim-obs, m)')

# Invert the y-axis
ax.invert_yaxis()
# save the figure in a graphics directory with the name 'residuals.png'
plt.savefig(os.path.join(dirGraphics,'residualsspatial_removewell.png'))

In [ ]:
#print the head of dfWells
print(dfWells.head())

## Write out Residuals as a Shapefile

In [ ]:
# Export your residuals as a shapefile
points_to_shapefile(dfWells, os.path.join(dirGraphics,nameOutHeadsPoints), outPrj=structure['header']['crs'])

## Plot the Residuals vs. Recharge

In [ ]:
# Plot residual vs. recharge
figResid, axResid = plt.subplots()
sns.regplot(x='rech', y='resid', data=dfWells, line_kws={'color':'red'})

# Add the 1:1 line <-- change the limits here to match your data
plt.plot([0.0001,dfWells['rech'].max()],[0,0], color='black', linestyle='--')
plt.xlim([0.0001,dfWells['rech'].max()]) #<-- change the limits here to match your data
plt.ylim([-50,200]) #<-- change the limits here to match your data
# Specify plot properties
axResid.set_xlabel('Input Recharge (m)',fontsize=12)
axResid.set_ylabel('Residual (sim-obs, m)',fontsize=12)
axResid.tick_params(axis='both', direction='in', right=True, top=True, length=6)
plt.title('Residual Plot, Sim-Obs Heads', fontsize=12)
plt.savefig(os.path.join(dirGraphics, 'residuals_recharge.png'), dpi=300)

## Plot the residuals within HK zones

In [ ]:
# For the purposes of visualization, drop rows within HK zones with fewer than 10 model cells and devide the number by 100
dfPlot = dfWells.groupby('hk').filter(lambda x: len(x) < 10)
dfPlot['hk'] = dfPlot['hk'].round(2) #Just to limited the number of decimal places

# Use seaborn to make a violin plot of residuals, split by `hk`
figViolin, axViolin = plt.subplots(figsize=[10,3])
sns.violinplot(x='hk', y='resid', data=dfPlot, ax=axViolin, hue='hk', palette=sns.color_palette("tab10"))
plt.plot(axViolin.get_xlim(),[0,0], color='black', linestyle=':')

# Update the legend, place it outside the box, limiting labels to %0.2g
plt.legend(title='HK (m/d)', loc='center left', bbox_to_anchor=(1, 0.5))
plt.savefig(os.path.join(dirGraphics, 'residuals_violin.png'), dpi=300)

In [ ]:
# For the purposes of visualization, drop rows within HK zones with fewer than 10 model cells and devide the number by 100
dfPlot = dfWells.groupby('hk').filter(lambda x: len(x) < 10)
dfPlot['hk'] = dfPlot['hk'].round(2) #Just to limited the number of decimal places

# Use seaborn to make a violin plot of residuals, split by `hk`
figViolin, axViolin = plt.subplots(figsize=[10,3])
sns.violinplot(x='hk', y='resid', data=dfPlot, ax=axViolin, hue='hk', palette=sns.color_palette("tab10"))
plt.plot(axViolin.get_xlim(),[0,0], color='black', linestyle=':')

# Update the legend, place it outside the box, limiting labels to %0.2g
plt.legend(title='HK (m/d)', loc='center left', bbox_to_anchor=(1, 0.5))
plt.savefig(os.path.join(dirGraphics, 'residuals_violin.png'), dpi=300)

In [ ]:
# plot hk vs recharge using seaborn scatter plot
figHKrech, axHKrech = plt.subplots()
sns.regplot(x='rech', y='hk', data=dfWells, line_kws={'color':'red'})
# Use seaborn to make a violin plot of residuals, split by `hk`
figViolin, axViolin = plt.subplots(figsize=[10,3])
sns.violinplot(x='hk', y='rech', data=dfPlot, ax=axViolin, hue='hk', palette=sns.color_palette("tab10"))
plt.plot(axViolin.get_xlim(),[0,0], color='black', linestyle=':')
plt.legend(title='HK (m/d)', loc='center left', bbox_to_anchor=(1, 0.5))
plt.savefig(os.path.join(dirGraphics, 'hk_recharge_comparison.png'), dpi=300)


In [ ]:
# plot hk vs recharge using seaborn scatter plot and divide hk within a cell
figHKrech, axHKrech = plt.subplots()
sns.regplot(x='rech', y='hk', data=dfWells, line_kws={'color':'black'})
plt.xlim([0.0001,dfWells['rech'].max()]) #<-- change the limits here to match your data
plt.ylim([0,80]) #<-- change the limits here to match your data
# Specify plot properties
axHKrech.set_xlabel('Input Recharge (m)',fontsize=12)
axHKrech.set_ylabel('HK (m/d)',fontsize=12)
axHKrech.tick_params(axis='both', direction='in', right=True, top=True, length=6)
plt.title('HK vs Recharge', fontsize=12)
plt.savefig(os.path.join(dirGraphics, 'hk_recharge.png'), dpi=300)

# Flows

## Compute the Flows Within Each Watershed

# Add the flows to dfDrn
dfFlows = dfDrn.merge(pd.DataFrame(flowData),left_index=True,right_index=True)
# extract the shape of watershed 

# Summarize the flows by watershed
dfWatershedFlows = dfWatersheds[['cellids','Watershed_ID']].merge(dfFlows,left_on='cellids',right_on='cellids').groupby('Watershed_ID').sum().reset_index()

# Keep only the fields we want
dfWatershedFlows = dfWatershedFlows[['Watershed_ID','q']]
dfWatershedFlows['q'] = -dfWatershedFlows['q'] #negative because the flow is out of the model

# Print out the output in cubic feet per second (my units are cubic meters per day)
dfWatershedFlows['q_cfs'] = dfWatershedFlows['q']*35.3147/86400
dfWatershedFlows['q_year'] = dfWatershedFlows['q']/86400*365.25